# Prompt Optimization: Improve a Prompt Automatically

[Read the cookbook page](https://docs.futureagi.com/docs/cookbook/quickstart/prompt-optimization).

## Set up Python

Use a fresh Python 3.12 kernel and an OpenAI key. Exact match runs locally.

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                'agent-learning-kit==0.3.7'], check=True)

In [ ]:
import os
from getpass import getpass

for name in ('OPENAI_API_KEY',):
    if not os.environ.get(name):
        value = getpass(f"{name}: ")
        if not value.strip():
            raise ValueError(f"{name} is required.")
        os.environ[name] = value.strip()
        del value

## Give the teacher a testable goal

Score two rounds on the same two labeled tickets; then try the selected prompt on a new ticket.

In [ ]:
from fi.evals.metrics import Equals
from fi.opt.base import Evaluator
from fi.opt.datamappers import BasicDataMapper
from fi.opt.generators import LiteLLMGenerator
from fi.opt.optimizers import MetaPromptOptimizer

MODEL = "openai/gpt-4o-mini"
seed = "Which queue, RETURNS or REPAIRS, handles this ticket? {ticket}"
train = [
    {"ticket": "My new boots don't fit. Can I send them back?", "queue": "RETURNS"},
    {"ticket": "Can you fix a broken buckle on my pack?", "queue": "REPAIRS"},
]
mapper = BasicDataMapper({
    "response": "generated_output",
    "expected_response": "queue",
})
evaluator = Evaluator(metric=Equals(config={"case_sensitive": True}))

teacher = LiteLLMGenerator(MODEL, "{prompt}")
optimizer = MetaPromptOptimizer(teacher_generator=teacher, task_model=MODEL)
result = optimizer.optimize(
    evaluator=evaluator, data_mapper=mapper, dataset=train, initial_prompts=[seed],
    task_description=("Route each ticket to RETURNS for refunds or REPAIRS for fixes. "
                      "Output only the uppercase queue name. Keep {ticket} as the only placeholder."),
    num_rounds=2, eval_subset_size=len(train),
)
for n, entry in enumerate(result.history, start=1):
    print(f"Round {n}: {entry.average_score:.0%}")
print("Selected prompt:", result.best_generator.get_prompt_template())
print("New ticket:", result.best_generator.generate({"ticket": "Can you repair a jacket seam?"}))

## Read the result

Compare the round scores and the new reply. The score is on the search examples, not on a held-out set. Confirm the selected template still includes `{ticket}` before reusing it.